# Cabeza por especie — versión final (nested CV + modelo final + métricas)

Tronco compartido + una cabeza por especie. Pipeline limpio y fundamentado:

**Nested CV (rendimiento honesto del método):** outer = folds homología globales (fold_1..5);
inner = Optuna sobre dropouts + lr + **alpha** (peso por especie), seleccionando por **macro-AUC core**
(las 6 especies curadas pesan igual; las extra entrenan el tronco pero no se usan para elegir).
Early stopping sobre validación interna del train, nunca sobre el fold de test.

**Métricas** por especie desde la OOF: AUROC, AUPRC, F1, MCC, Precision, Recall, Accuracy (umbral 0.5),
con **IC95% por bootstrap** (sin semillas: el ± sale del remuestreo de la propia OOF).

**Modelo final:** búsqueda de hiperparámetros sobre TODO el dataset → reentrenar → guardar. No se evalúa
sobre sí mismo; su rendimiento esperado lo da el nested.

**Fuga:** la celda 3b avisa de IDs compartidos entre especies (pendiente: clustering homología global).


In [ ]:
# 1. Mount Drive + imports
from google.colab import drive
drive.mount('/content/drive')

import os, gc, json, random
import numpy as np, pandas as pd
import tensorflow as tf, tensorflow.keras as ks
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             matthews_corrcoef, precision_score, recall_score, accuracy_score)
!pip install optuna -q
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
print('TF', tf.__version__)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 20.9 MB/s eta 0:00:00
TF 2.20.0


In [ ]:
# 2. Configuration
BASE       = '/content/drive/MyDrive/IMMUNO_MODEL/DEFINITIVE_CODES'
NUEVA_BASE = '/content/drive/MyDrive/IMMUNO_MODEL/INTEGRATED/nueva prueba'

CORE_PATHS = {
    'Bos_taurus':     f'{BASE}/Bos_taurus/embeddings.npz',
    'Gallus_gallus':  f'{BASE}/Gallus_gallus/embeddings.npz',
    'Homo_sapiens':   f'{BASE}/Homo_sapiens/embeddings.npz',
    'Equus_caballus': f'{BASE}/Equus/embeddings.npz',
    'Sus_scrofa':     f'{BASE}/Sus_scrofa/embeddings.npz',
    'Canis_sp':       f'{BASE}/Canis/embeddings.npz',
}
NEW_SPECIES = ['Anseriformes','Camelidae','Capra_hircus','Cavia_porcellus','Felis_catus',
               'Fish','Macaca_sp_','Mus_musculus','Non_human_primate','Oryctolagus_cuniculus',
               'Ovis_aries','Rattus_sp_']
NEW_PATHS = {s: f'{NUEVA_BASE}/{s}/embeddings.npz' for s in NEW_SPECIES}
EXCLUDE_SPECIES = ['Anseriformes', 'Felis_catus', 'Fish']

TRUNK_UNITS = [1024, 256, 64]
EPOCHS, BATCH_SIZE, PATIENCE = 500, 256, 10
N_TRIALS_INNER = 15
N_TRIALS_FINAL = 25
SEED = 42

OUTDIR = f'{BASE}/INTEGRATED/cabeza_por_especie_FINAL'
FIGDIR = f'{OUTDIR}/figuras'
os.makedirs(FIGDIR, exist_ok=True)
MODEL_PATH = f'{OUTDIR}/modelo_final.keras'

ALL_PATHS = {**CORE_PATHS, **NEW_PATHS}
CORE_SET  = set(CORE_PATHS)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('Resultados ->', OUTDIR)

Resultados -> /content/drive/MyDrive/IMMUNO_MODEL/DEFINITIVE_CODES/INTEGRATED/cabeza_por_especie_FINAL


In [ ]:
# 3. Load the hosts (minus the excluded ones) and merge
def load_npz(path):
    d = np.load(path, allow_pickle=True)
    y  = d['labels'].astype(int); X = d['X'].astype('float32')
    cv = d['cv_homology'].astype(str) if 'cv_homology' in d else np.array(['NA']*len(y))
    ids = d['ids'].astype(str) if 'ids' in d else np.array([f'{os.path.basename(path)}_{i}' for i in range(len(y))])
    return X, y, cv, ids

X_list, y_list, sp_list, cv_list, id_list = [], [], [], [], []
for h, p in ALL_PATHS.items():
    if h in EXCLUDE_SPECIES: print(f'  [EXCLUIDA] {h}'); continue
    if not os.path.exists(p): print(f'  !! FALTA {h}'); continue
    X, y, cv, ids = load_npz(p)
    X_list.append(X); y_list.append(y); cv_list.append(cv); id_list.append(ids)
    sp_list.append(np.array([h]*len(y)))
    print(f'  {h:22s} n={len(y):5d}  pos={int(y.sum()):5d}  {"core" if h in CORE_SET else "nueva"}')

X_all=np.concatenate(X_list).astype('float32'); y_all=np.concatenate(y_list).astype(int)
species_all=np.concatenate(sp_list); cv_all=np.concatenate(cv_list).astype(str); ids_all=np.concatenate(id_list).astype(str)
SPECIES=list(dict.fromkeys(species_all)); N_SP=len(SPECIES)
sp2id={s:i for i,s in enumerate(SPECIES)}; sp_id_all=np.array([sp2id[s] for s in species_all],dtype='int32')
CORE_SPECIES=[s for s in SPECIES if s in CORE_SET]
is_core=np.array([s in CORE_SET for s in species_all])
print(f'\nTOTAL X={X_all.shape} | especies={N_SP} (core={len(CORE_SPECIES)})')

  Bos_taurus             n=  882  pos=  441  core
  Gallus_gallus          n=  920  pos=  460  core
  Homo_sapiens           n= 7035  pos= 3562  core
  Equus_caballus         n=  670  pos=  335  core
  Sus_scrofa             n= 1056  pos=  528  core
  Canis_sp               n=  704  pos=  352  core
  [EXCLUIDA] Anseriformes
  Camelidae              n=  452  pos=  226  nueva
  Capra_hircus           n=   94  pos=   47  nueva
  Cavia_porcellus        n=  124  pos=   62  nueva
  [EXCLUIDA] Felis_catus
  [EXCLUIDA] Fish
  Macaca_sp_             n=  248  pos=  124  nueva
  Mus_musculus           n= 7536  pos= 3768  nueva
  Non_human_primate      n=  112  pos=   56  nueva
  Oryctolagus_cuniculus  n= 1654  pos=  827  nueva
  Ovis_aries             n=  160  pos=   80  nueva
  Rattus_sp_             n=  316  pos=  158  nueva

TOTAL X=(21963, 1280) | especies=15 (core=6)


In [ ]:
# 3b. Check for IDs shared between hosts (potential leakage through the trunk)
df_id = pd.DataFrame({'id': ids_all, 'species': species_all})
dup = df_id.groupby('id')['species'].nunique(); dup = dup[dup > 1]
print(f'IDs en >1 especie: {len(dup)}')
if len(dup):
    print('>> Hay solape de IDs entre especies -> con tronco compartido es FUGA parcial.')
    print('   Los folds por especie no lo evitan; arreglo definitivo = clustering homología GLOBAL (pendiente).')
else:
    print('>> Sin solape de IDs entre especies.')

IDs en >1 especie: 671
>> Hay solape de IDs entre especies -> con tronco compartido es FUGA parcial.
   Los folds por especie no lo evitan; arreglo definitivo = clustering homología GLOBAL (pendiente).


In [ ]:
# 4. Model (shared trunk + one head per host), weights and metrics
def set_seed(s): random.seed(s); np.random.seed(s); tf.random.set_seed(s)

def build_multihead(norm, cfg):
    emb_in = ks.Input((1280,), name='esm2_embedding'); sp_in = ks.Input((1,), dtype='int32', name='species_id')
    x = norm(emb_in)
    for i,(u,d) in enumerate(zip(cfg['units'], cfg['dropout']), start=1):
        x = ks.layers.Dense(u, activation='relu', name=f'trunk{i}_dense')(x)
        x = ks.layers.Dropout(d)(x)
    heads = ks.layers.Dense(N_SP, activation='sigmoid', name='species_heads')(x)
    mask  = ks.layers.Lambda(lambda i: tf.one_hot(tf.reshape(i,[-1]), N_SP), output_shape=(N_SP,), name='mask')(sp_in)
    out = ks.layers.Lambda(lambda a: tf.reduce_sum(a[0]*a[1], axis=1, keepdims=True), output_shape=(1,), name='pick')([heads, mask])
    m = ks.Model([emb_in, sp_in], out)
    m.compile(optimizer=ks.optimizers.Adam(cfg['lr']), loss='binary_crossentropy', metrics=['accuracy'])
    return m

def make_sw(y_tr, sp_tr, alpha):
    classes = np.unique(y_tr)
    cw = dict(zip(classes, compute_class_weight('balanced', classes=classes, y=y_tr)))
    class_w = np.array([cw[v] for v in y_tr])
    sps, counts = np.unique(sp_tr, return_counts=True); n = len(sps)
    wmap = {s: (len(sp_tr)/(n*c))**alpha for s, c in zip(sps, counts)}
    return (class_w * np.array([wmap[s] for s in sp_tr])).astype('float32')

def fit_multihead(Xtr, ytr, spid_tr, spn_tr, Xva, yva, spid_va, cfg):
    sw = make_sw(ytr, spn_tr, cfg.get('alpha', 1.0))
    norm = ks.layers.Normalization(); norm.adapt(Xtr)
    m = build_multihead(norm, cfg)
    m.fit([Xtr, spid_tr], ytr, validation_data=([Xva, spid_va], yva), epochs=EPOCHS, batch_size=BATCH_SIZE,
          sample_weight=sw, callbacks=[ks.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True)], verbose=0)
    return m

def auc_safe(yt, ys): return roc_auc_score(yt, ys) if len(np.unique(yt)) > 1 else np.nan
def macro_auc_core(yt, ys, spn):
    v = [auc_safe(yt[spn==s], ys[spn==s]) for s in CORE_SPECIES if (spn==s).any()]
    return np.nanmean(v) if v else np.nan

def all_metrics(yt, ys, thr=0.5):
    if len(yt)==0 or len(np.unique(yt))<2: return {k:np.nan for k in ['AUROC','AUPRC','F1','MCC','Precision','Recall','Accuracy']}
    yp=(ys>=thr).astype(int)
    return {'AUROC':roc_auc_score(yt,ys),'AUPRC':average_precision_score(yt,ys),'F1':f1_score(yt,yp,zero_division=0),
            'MCC':matthews_corrcoef(yt,yp),'Precision':precision_score(yt,yp,zero_division=0),
            'Recall':recall_score(yt,yp,zero_division=0),'Accuracy':accuracy_score(yt,yp)}

def boot_ci_auc(yt, ys, B=1000, seed=0):
    rng=np.random.default_rng(seed); n=len(yt); vals=[]
    for _ in range(B):
        ii=rng.integers(0,n,n)
        if len(np.unique(yt[ii]))>1: vals.append(roc_auc_score(yt[ii], ys[ii]))
    return (np.percentile(vals,2.5), np.percentile(vals,97.5)) if vals else (np.nan,np.nan)
print('modelo y utilidades listos. N_SP =', N_SP)

modelo y utilidades listos. N_SP = 15


In [ ]:
# 5. Nested CV (inner = Optuna over dropouts, lr, alpha; selected by core macro-AUC)
def inner_search(idx_tr, cv_tr, folds_tr, n_trials, fold_label='', seed=SEED):
    val_g = folds_tr[0]; vi = cv_tr == val_g; ti = ~vi
    Xti,yti, spid_ti,spn_ti = X_all[idx_tr][ti], y_all[idx_tr][ti], sp_id_all[idx_tr][ti], species_all[idx_tr][ti]
    Xvi,yvi,spid_vi,spn_vi = X_all[idx_tr][vi], y_all[idx_tr][vi], sp_id_all[idx_tr][vi], species_all[idx_tr][vi]
    def objective(trial):
        cfg = {'units': TRUNK_UNITS,
               'dropout': [trial.suggest_float('d1',0.2,0.7), trial.suggest_float('d2',0.3,0.7), trial.suggest_float('d3',0.2,0.6)],
               'lr': trial.suggest_float('lr',1e-5,1e-3,log=True),
               'alpha': trial.suggest_float('alpha',0.0,1.0)}
        m = fit_multihead(Xti,yti,spid_ti,spn_ti, Xvi,yvi,spid_vi, cfg)
        val = macro_auc_core(yvi, m.predict([Xvi,spid_vi],verbose=0).ravel(), spn_vi)
        ks.backend.clear_session(); gc.collect()
        return val if np.isfinite(val) else 0.0
    st = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=seed))
    st.optimize(objective, n_trials=n_trials)
    return st.best_params

def nested_cv(n_trials=N_TRIALS_INNER):
    set_seed(SEED)
    idx_full = np.arange(len(y_all)); folds = [f for f in np.unique(cv_all) if f != 'NA']
    oof_t, oof_s, oof_i, chosen = [], [], [], []
    for k, g in enumerate(folds, 1):
        print(f'\n=== fold externo {k}/{len(folds)} ({g}) ===', flush=True)
        te = cv_all==g; tr = (cv_all!=g)&(cv_all!='NA')
        idx_tr = idx_full[tr]; cv_tr = cv_all[tr]; folds_tr = [f for f in folds if f != g]
        best = inner_search(idx_tr, cv_tr, folds_tr, n_trials, fold_label=g)
        cfg = {'units':TRUNK_UNITS, 'dropout':[best['d1'],best['d2'],best['d3']], 'lr':best['lr'], 'alpha':best['alpha']}
        inner_val = folds_tr[0]; vi = cv_all==inner_val; trn = tr & (cv_all!=inner_val)
        m = fit_multihead(X_all[trn],y_all[trn],sp_id_all[trn],species_all[trn], X_all[vi],y_all[vi],sp_id_all[vi], cfg)
        s = m.predict([X_all[te], sp_id_all[te]], verbose=0).ravel()
        oof_t.append(y_all[te]); oof_s.append(s); oof_i.append(idx_full[te]); chosen.append({'fold':g, **best})
        print(f'  cfg={best}', flush=True); ks.backend.clear_session(); gc.collect()
    return np.concatenate(oof_t), np.concatenate(oof_s), np.concatenate(oof_i), pd.DataFrame(chosen)
print('nested_cv listo')

nested_cv listo


In [ ]:
# 6. Run the nested CV -> per-host metrics (bootstrap 95% CI) + save the CSVs
yt, ys, idx, chosen = nested_cv()
sp_oof = species_all[idx]

rows = []
for s in SPECIES:
    m = sp_oof == s
    met = all_metrics(yt[m], ys[m])
    lo, hi = boot_ci_auc(yt[m], ys[m])
    rows.append({'especie': s, 'tipo': 'core' if s in CORE_SET else 'nueva', 'n': int(m.sum()),
                 **met, 'AUROC_IC95_lo': lo, 'AUROC_IC95_hi': hi})
per_sp = pd.DataFrame(rows).set_index('especie')

summary = pd.DataFrame({
    'macro_AUROC_core': [np.nanmean(per_sp.loc[CORE_SPECIES,'AUROC'])],
    'macro_AUROC_all':  [np.nanmean(per_sp['AUROC'])],
    'pooled_AUROC':     [auc_safe(yt, ys)],
    'pooled_AUPRC':     [average_precision_score(yt, ys)],
}).T.rename(columns={0:'valor'})

print('=== NESTED — resumen ==='); display(summary.round(4))
print('=== por especie ==='); display(per_sp.round(4))

per_sp.to_csv(f'{OUTDIR}/metricas_por_especie_NESTED.csv')
summary.to_csv(f'{OUTDIR}/summary_NESTED.csv')
chosen.to_csv(f'{OUTDIR}/hiperparametros_por_fold.csv', index=False)
pd.DataFrame({'id':ids_all[idx],'species':sp_oof,'is_core':is_core[idx],'y_true':yt,'y_score':ys}) \
  .to_csv(f'{OUTDIR}/oof_predicciones.csv', index=False)
print('CSVs guardados en', OUTDIR)


=== fold externo 1/5 (fold_1) ===
  cfg={'d1': 0.2917022549267169, 'd2': 0.4216968971838151, 'd3': 0.4099025726528951, 'lr': 7.309539835912905e-05, 'alpha': 0.2912291401980419}

=== fold externo 2/5 (fold_2) ===
  cfg={'d1': 0.2610191174223894, 'd2': 0.498070764044508, 'd3': 0.21375540844608737, 'lr': 0.000658628931758311, 'alpha': 0.2587799816000169}

=== fold externo 3/5 (fold_3) ===
  cfg={'d1': 0.3872700594236812, 'd2': 0.6802857225639665, 'd3': 0.49279757672456204, 'lr': 0.00015751320499779721, 'alpha': 0.15601864044243652}

=== fold externo 4/5 (fold_4) ===
  cfg={'d1': 0.41457386362653975, 'd2': 0.5364224779557715, 'd3': 0.25829641438481477, 'lr': 0.00036222921641170983, 'alpha': 0.4874999513303224}

=== fold externo 5/5 (fold_5) ===
  cfg={'d1': 0.5312611421769909, 'd2': 0.42468443043576437, 'd3': 0.4080272084711243, 'lr': 0.00012399967836846095, 'alpha': 0.18485445552552704}
=== NESTED — resumen ===


,valor
macro_AUROC_core,0.8612
macro_AUROC_all,0.8947
pooled_AUROC,0.9161
pooled_AUPRC,0.9221


=== por especie ===


,tipo,n,AUROC,AUPRC,F1,MCC,Precision,Recall,Accuracy,AUROC_IC95_lo,AUROC_IC95_hi
especie,,,,,,,,,,,
Bos_taurus,core,882,0.8262,0.8364,0.7390,0.5199,0.7995,0.6871,0.7574,0.8003,0.8512
Gallus_gallus,core,920,0.8368,0.8338,0.7661,0.5441,0.7854,0.7478,0.7717,0.8098,0.8620
Homo_sapiens,core,7035,0.9266,0.9397,0.8596,0.7250,0.8852,0.8355,0.8618,0.9205,0.9327
Equus_caballus,core,670,0.8998,0.8732,0.8128,0.6333,0.8292,0.7970,0.8164,0.8756,0.9224
Sus_scrofa,core,1056,0.8696,0.8719,0.7703,0.5774,0.8311,0.7178,0.7860,0.8482,0.8910
Canis_sp,core,704,0.8083,0.8191,0.7319,0.4875,0.7647,0.7017,0.7429,0.7799,0.8397
Camelidae,nueva,452,0.8738,0.8758,0.8028,0.6266,0.8439,0.7655,0.8119,0.8404,0.9063
Capra_hircus,nueva,94,0.9000,0.9184,0.8352,0.6822,0.8636,0.8085,0.8404,0.8280,0.9620
Cavia_porcellus,nueva,124,0.9594,0.9711,0.9355,0.8710,0.9355,0.9355,0.9355,0.9198,0.9901


CSVs guardados en /content/drive/MyDrive/IMMUNO_MODEL/DEFINITIVE_CODES/INTEGRATED/cabeza_por_especie_FINAL


In [ ]:
# 7. Final model: search over the WHOLE dataset, retrain, save
set_seed(SEED)
idx_full = np.arange(len(y_all)); folds = [f for f in np.unique(cv_all) if f != 'NA']
best_final = inner_search(idx_full[cv_all!='NA'], cv_all[cv_all!='NA'], folds, N_TRIALS_FINAL, fold_label='FINAL')
print('Hiperparámetros del modelo final:', best_final)

cfg_f = {'units':TRUNK_UNITS, 'dropout':[best_final['d1'],best_final['d2'],best_final['d3']],
         'lr':best_final['lr'], 'alpha':best_final['alpha']}
val_g = folds[0]; vi = cv_all==val_g; trn = (cv_all!='NA') & (cv_all!=val_g)   # val interna solo para early stopping
final_model = fit_multihead(X_all[trn],y_all[trn],sp_id_all[trn],species_all[trn],
                            X_all[vi],y_all[vi],sp_id_all[vi], cfg_f)
final_model.save(MODEL_PATH)
with open(f'{OUTDIR}/config_modelo_final.json','w') as f:
    json.dump({**cfg_f, 'species_order': SPECIES, 'sp2id': sp2id}, f, indent=2)
print('Modelo final guardado en', MODEL_PATH)
print('NOTA: el modelo final NO se evalúa sobre sí mismo; su rendimiento esperado es el del nested (celda 6).')

Hiperparámetros del modelo final: {'d1': 0.45272965888280087, 'd2': 0.3025315911388134, 'd3': 0.36299240299823066, 'lr': 0.00011308393990141176, 'alpha': 0.24572295830227647}
Modelo final guardado en /content/drive/MyDrive/IMMUNO_MODEL/DEFINITIVE_CODES/INTEGRATED/cabeza_por_especie_FINAL/modelo_final.keras
NOTA: el modelo final NO se evalúa sobre sí mismo; su rendimiento esperado es el del nested (celda 6).
